# Network Intrusion Detection — NNConv Link Predictor
Full pipeline: data loading → subsampling → edge split → negative sampling → PyG Data → NNConv model → evaluation

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import NNConv

from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Helper functions — parse GraphML

In [ ]:
def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph."""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)


def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except Exception:
                    edge_attrs[attr_name] = data_elem.text
        G.add_edge(source, target, **edge_attrs)

    return G


def extract_edge_features(G):
    """Extract edge features as a pandas DataFrame."""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {'source': u, 'target': v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Load graph

In [ ]:
FILE_PATH = "/Users/dannykim/Desktop/vscode/Comp 395/Network-Intrusion-GNN/archive/0.1M-Stratified-Multi.graphml"

G_full = parse_graphml(FILE_PATH)
print(f"Full graph  — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")

# Extract edge features from the full graph for later use
edge_df = extract_edge_features(G_full)
print("\nEdge feature columns:", edge_df.columns.tolist())
print(edge_df.describe())

## 4. Subsample to a manageable size

The full graph has 41k nodes. Using a one-hot or degree feature matrix on that is expensive,
so we subsample to ~2000 nodes / 8000 edges for development.

In [ ]:


def get_dense_subgraph(G_full, target_nodes=2000):
    # 1. Pick a random starting node that actually has connections
    random.seed(SEED)
    start_node = random.choice([n for n, d in G_full.degree() if d > 5])
    
    # 2. Use BFS to find the closest 2000 nodes
    nodes = {start_node}
    queue = [start_node]
    
    while len(nodes) < target_nodes and queue:
        current = queue.pop(0)
        for neighbor in G_full.neighbors(current):
            if neighbor not in nodes:
                nodes.add(neighbor)
                queue.append(neighbor)
                if len(nodes) >= target_nodes:
                    break
                    
    # 3. Create the induced subgraph
    return G_full.subgraph(nodes).copy()

# Update your Section 4 with this:
G_sub = get_dense_subgraph(G_full, target_nodes=3000)
G_int = nx.convert_node_labels_to_integers(G_sub)

print(f"Subgraph — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")

print(f"Relabelled  — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

## visualize graph

In [ ]:


def visualize_split(G_int, train_edges, val_edges, test_edges):
    plt.figure(figsize=(10, 7))
    pos = nx.spring_layout(G_int, seed=42) # Consistent layout
    
    # Draw nodes
    nx.draw_networkx_nodes(G_int, pos, node_size=20, node_color='lightgrey', alpha=0.8)
    
    # Draw edges with different colors
    nx.draw_networkx_edges(G_int, pos, edgelist=train_edges, edge_color='blue', label='Train', width=1, alpha=0.5)
    nx.draw_networkx_edges(G_int, pos, edgelist=val_edges, edge_color='orange', label='Val', width=1.5)
    nx.draw_networkx_edges(G_int, pos, edgelist=test_edges, edge_color='red', label='Test', width=1.5)
    
    plt.title("Graph Edge Split Visualization")
    plt.legend()
    plt.show()

# Run the visualization
visualize_split(G_int, train_edges, val_edges, test_edges)

## 5. Edge train / val / test split

In [ ]:
def preview_graph_edges(G, name, n=5):
    edges = list(G.edges(data=True))[:n]
    print(f"\n--- {name} (first {n} edges) ---")
    for u, v, edata in edges:
        # MultiGraph: edata is {0: {attrs}, 1: {attrs}, ...}
        # Grab the first parallel edge's attributes
        inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
        label = inner.get('ActivityLabel', 'N/A')
        print(f"  ({u}, {v}) → ActivityLabel: {label}")

In [ ]:
# all_edges = list(G_int.edges())
# random.shuffle(all_edges)
# n = len(all_edges)

# train_edges = all_edges[:int(0.70 * n)]   # 70%
# val_edges   = all_edges[int(0.70 * n):int(0.85 * n)]  # 15%
# test_edges  = all_edges[int(0.85 * n):]   # 15%

# print(f"Train edges: {len(train_edges):,}")
# print(f"Val edges:   {len(val_edges):,}")
# print(f"Test edges:  {len(test_edges):,}")

# # Training graph — only contains train edges to prevent data leakage
# G_train = nx.Graph()
# G_train.add_nodes_from(G_int.nodes(data=True))
# G_train.add_edges_from(
#     (u, v, G_int[u][v]) for u, v in train_edges
# )




# 1. Get all current edges and shuffle them
# all_edges = list(G_int.edges())
# random.seed(SEED)
# random.shuffle(all_edges)

# n = len(all_edges)

# # 2. Define the split indices #TODO
# train_split = int(0.70 * n)
# val_split   = int(0.85 * n)

# # 3. Carve out the edge sets
# train_edges = all_edges[:train_split]   # 70% - These stay in the graph
# val_edges   = all_edges[train_split:val_split]  # 15% - "Hidden" for tuning
# test_edges  = all_edges[val_split:]   # 15% - "Hidden" for final exam

# print(f"Train edges: {len(train_edges):,}")
# print(f"Val edges:   {len(val_edges):,}")
# print(f"Test edges:  {len(test_edges):,}")

# # 4. Create the Training Graph (The Message-Passing Backbone)
# # Crucial: This graph ONLY contains train_edges. 
# # If a Val or Test edge is in here, it's DATA LEAKAGE.
# G_train = nx.MultiGraph()
# G_train.add_nodes_from(G_int.nodes(data=True)) # Keep all 3,000 nodes

# # Add only the training edges, including their traffic features
# G_train.add_edges_from(
#     (u, v, G_int[u][v]) for u, v in train_edges
# )

# print(f"G_train nodes: {G_train.number_of_nodes():,} edges: {G_train.number_of_edges():,}")




# ── Semi-Supervised Anomaly Detection Split ───────────────────────────────────
normal_edges = []
attack_edges = []

for u, v, edata in G_int.edges(data=True):
    label = edata.get('ActivityLabel')  # adjust key if yours differs
    if label == 0:
        normal_edges.append((u, v))
    else:
        attack_edges.append((u, v))

print(f"Normal edges: {len(normal_edges):,}")
print(f"Attack edges: {len(attack_edges):,}")

# # Step 2: Split normal edges into train / val / test
random.seed(SEED)
random.shuffle(normal_edges)

n = len(normal_edges)
train_split = int(0.70 * n)
val_split   = int(0.85 * n)

train_edges        = normal_edges[:train_split]          # 70% normal — training only
val_edges          = normal_edges[train_split:val_split] # 15% normal — hyperparameter tuning
test_normal_edges  = normal_edges[val_split:]            # 15% normal — held-out for test
test_attack_edges  = attack_edges                        # ALL attacks — for test only

# # Step 3: Build the test set (normal + attack, for evaluation)

test_edges  = test_normal_edges + test_attack_edges
# test_labels = [0] * len(test_normal_edges) + [1] * len(test_attack_edges)

print(f"\nTrain edges (normal only): {len(train_edges):,}")
print(f"Val edges     (normal only): {len(val_edges):,}")
print(f"Test normal:  (normal only): {len(test_normal_edges):,}")
print(f"Test attack:  (normal + attack): {len(test_attack_edges):,}")
print(f"Test set anomaly ratio:    {len(test_attack_edges)/len(test_edges):.2%}")

# # Step 4: Build the training graph (normal edges only — no data leakage)
G_train = nx.MultiGraph()
G_train.add_nodes_from(G_int.nodes(data=True))
G_train.add_edges_from(
    (u, v, G_int[u][v]) for u, v in train_edges
)

G_val = nx.MultiGraph()
G_val.add_nodes_from(G_int.nodes(data=True))
G_val.add_edges_from(
    (u, v, G_int[u][v]) for u, v in val_edges
)


print(f"\nG_train — nodes: {G_train.number_of_nodes():,}  edges: {G_train.number_of_edges():,}")
preview_graph_edges(G_train, "G_train")
print(f"G_val — nodes: {G_val.number_of_nodes():,}  edges: {G_val.number_of_edges():,}")
preview_graph_edges(G_val,   "G_val")




## 6. Negative edge sampling

For each split we generate an equal number of fake (non-existent) edges.
These act as the negative class for the binary classification problem.

In [ ]:
# def generate_negative_edges(G, num_samples, seed=42):
#     """Sample edges that do NOT exist in G."""
#     random.seed(seed)
#     neg = []
#     nodes = list(G.nodes())
#     existing = set(G.edges()) | {(v, u) for u, v in G.edges()}
#     while len(neg) < num_samples:
#         u, v = random.sample(nodes, 2)
#         if (u, v) not in existing:
#             neg.append((u, v))
#     return neg


# neg_train = generate_negative_edges(G_train, len(train_edges), seed=42)
# neg_val   = generate_negative_edges(G_train, len(val_edges),   seed=43)
# neg_test  = generate_negative_edges(G_train, len(test_edges),  seed=44)

# print(f"Negative train: {len(neg_train):,}")
# print(f"Negative val:   {len(neg_val):,}")
# print(f"Negative test:  {len(neg_test):,}")

## 7. Build node index map and integer edge lists

`node2idx` maps each node ID to a row index in the feature matrix.
All edge lists must use these integer indices for PyG.

In [ ]:
# node2idx: node_id -> integer index
node2idx = {n: i for i, n in enumerate(G_train.nodes())}


def to_idx(edges):
    """Convert a list of (u, v) node-ID pairs to integer index pairs.
    Drops any edge whose endpoints are not in node2idx (e.g. isolated val/test nodes).
    """
    return [
        (node2idx[u], node2idx[v])
        for u, v in edges
        if u in node2idx and v in node2idx
    ]


train_pos_idx = to_idx(train_edges)
val_pos_idx   = to_idx(val_edges)

test_normal_idx = to_idx(test_normal_edges)
test_attack_idx = to_idx(test_attack_edges)


print(f"train_pos_idx: {len(train_pos_idx):,}")
print(f"val_pos_idx:   {len(val_pos_idx):,}")
print(f"test_normal_idx:  {len(test_normal_idx):,}")
print(f"test_attack_idx:  {len(test_attack_idx):,}")



## 8. Build PyG Data object with edge features

Node features: normalised degree (1 scalar per node — cheap and informative).
Edge features: the 7 traffic columns from the dataset.

In [ ]:
# EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
#                   'Proto_encoded', 'Dir_encoded', 'State_encoded']
# NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7


# # ── Node features: normalised degree ──────────────────────────────────────────
# degrees = np.array([G_train.degree(n) for n in G_train.nodes()], dtype=float)
# degrees = (degrees - degrees.mean()) / (degrees.std() + 1e-9)
# x = torch.tensor(degrees, dtype=torch.float).unsqueeze(1)  # shape [N, 1]

# # ── Edge feature lookup: (u, v) -> feature vector ─────────────────────────────
# # We build this from edge_df (extracted from the full graph earlier).
# # For edges missing from edge_df we fall back to zeros.
# edge_feat_dict = {}
# zero_feats = np.zeros(NUM_EDGE_FEATS)

# for _, row in edge_df.iterrows():
#     u_raw, v_raw = row['source'], row['target']
#     # edge_df uses original string node IDs; we need integer IDs from G_int
#     # G_int was relabelled from G_sub, so we look up via the mapping stored
#     # inside G_sub's relabelling.  Simplest approach: check both directions.
#     feats = []
#     for col in EDGE_FEAT_COLS:
#         val = row.get(col, 0.0)
#         feats.append(float(val) if pd.notna(val) else 0.0)
#     feats = np.array(feats)
#     edge_feat_dict[(u_raw, v_raw)] = feats
#     edge_feat_dict[(v_raw, u_raw)] = feats

# # ── Edge index and edge_attr for the training graph ───────────────────────────
# train_edge_list = list(G_train.edges())

# edge_index = torch.tensor(
#     [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
#     dtype=torch.long
# ).t().contiguous()  # shape [2, E]

# edge_attr = []
# for u, v in train_edge_list:
#     # G_train stores integer node IDs; edge_feat_dict has original string IDs.
#     # G_int carries the original label in node attribute 'label' if available,
#     # but simplest fallback: use the raw edge data stored on G_train itself.
#     raw_data = G_train[u][v]
#     feats = []
#     for col in EDGE_FEAT_COLS:
#         val = raw_data.get(col, 0.0)
#         feats.append(float(val) if val is not None else 0.0)
#     edge_attr.append(feats)

# edge_attr = torch.tensor(edge_attr, dtype=torch.float)  # shape [E, 7]

# data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# print("Data object:")
# print(f"  x:          {data.x.shape}")
# print(f"  edge_index: {data.edge_index.shape}")
# print(f"  edge_attr:  {data.edge_attr.shape}")

# Aggregate edge features per node as node features
# For each node, compute mean of its edges' traffic values




EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']


NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7



node_feat_dict = {n: [] for n in G_train.nodes()}
for u, v, edata in G_train.edges(data=True):
    for key in edata:  # key = 0, 1, 2... for each parallel edge
        inner = edata[key]
        feats = [float(inner.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
        node_feat_dict[u].append(feats)
        node_feat_dict[v].append(feats)

node_feats = []
for n in G_train.nodes():
    edges_feats = node_feat_dict[n]
    if edges_feats:
        node_feats.append(np.mean(edges_feats, axis=0))
    else:
        node_feats.append(np.zeros(len(EDGE_FEAT_COLS)))

node_feats = np.array(node_feats)

# Normalize
node_feats = (node_feats - node_feats.mean(axis=0)) / (node_feats.std(axis=0) + 1e-9)
x = torch.tensor(node_feats, dtype=torch.float)  # shape [N, 7]


# ── Edge feature lookup: (u, v) -> feature vector ─────────────────────────────
# We build this from edge_df (extracted from the full graph earlier).
# For edges missing from edge_df we fall back to zeros.
edge_feat_dict = {}
zero_feats = np.zeros(NUM_EDGE_FEATS)

for _, row in edge_df.iterrows():
    u_raw, v_raw = row['source'], row['target']
    # edge_df uses original string node IDs; we need integer IDs from G_int
    # G_int was relabelled from G_sub, so we look up via the mapping stored
    # inside G_sub's relabelling.  Simplest approach: check both directions.
    feats = []
    for col in EDGE_FEAT_COLS:
        val = row.get(col, 0.0)
        feats.append(float(val) if pd.notna(val) else 0.0)
    feats = np.array(feats)
    edge_feat_dict[(u_raw, v_raw)] = feats
    edge_feat_dict[(v_raw, u_raw)] = feats

# ── Edge index and edge_attr for the training graph ───────────────────────────
train_edge_list = list(G_train.edges())

edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
    dtype=torch.long
).t().contiguous()  # shape [2, E]

edge_attr = []
for u, v in train_edge_list:
    raw_data = G_train[u][v][0]
    feats = [float(raw_data.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    edge_attr.append(feats)

edge_attr = torch.tensor(edge_attr, dtype=torch.float)  # shape [E, 7]

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

print("Data object:")
print(f"  x:          {data.x.shape}")
print(f"  edge_index: {data.edge_index.shape}")
print(f"  edge_attr:  {data.edge_attr.shape}")


## 9. NNConv model

NNConv uses edge features to compute a per-edge weight matrix that transforms
the source node's features before aggregating into the target node.
This means every edge contributes differently to the message passing,
which is exactly what we want when traffic features carry the signal.

In [ ]:
HIDDEN = 64


class GNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # Conv 1: node_in -> hidden
        # edge_nn output must equal node_in * hidden (produces the weight matrix)
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )

        # Conv 2: hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )

        # MLP decoder: concat of two node embeddings -> edge score
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def encode(self, x, edge_index, edge_attr):
        """Run message passing to produce node embeddings."""
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.conv2(x, edge_index, edge_attr)
        return x

    def decode(self, z, edges):
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)
        # Dot product similarity — higher = more normal
        return (z[u] * z[v]).sum(dim=1)

    def forward(self, x, edge_index, edge_attr, edges):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges)


model = GNN(
    node_in=x.shape[1],
    edge_in=NUM_EDGE_FEATS,
    hidden=HIDDEN
)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## 10. Training

In [ ]:
def evaluate(model, data, val_normal_idx):
    """Validation: just check normal edge scores are high (sanity check)."""
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)
        u = torch.tensor([e[0] for e in val_normal_idx], dtype=torch.long)
        v = torch.tensor([e[1] for e in val_normal_idx], dtype=torch.long)
        scores = torch.sigmoid((z[u] * z[v]).sum(dim=1))
    print(f"  Val normal edge score (mean): {scores.mean():.4f}")
    return scores.mean().item()


def evaluate_test(model, data, test_normal_idx, test_attack_idx):
    """Final test: AUROC and AUPRC separating normal from attack edges."""
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)

        all_edges = test_normal_idx + test_attack_idx
        u = torch.tensor([e[0] for e in all_edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in all_edges], dtype=torch.long)
        scores = torch.sigmoid((z[u] * z[v]).sum(dim=1)).numpy()

    # Anomaly score = 1 - similarity (low similarity = more anomalous)
    anomaly_scores = 1 - scores
    y_true = [0] * len(test_normal_idx) + [1] * len(test_attack_idx)

    auc  = roc_auc_score(y_true, anomaly_scores)
    ap   = average_precision_score(y_true, anomaly_scores)
    return auc, ap

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    z = model.encode(data.x, data.edge_index, data.edge_attr)

    # Score all training edges (normal only)
    u = torch.tensor([e[0] for e in train_pos_idx], dtype=torch.long)
    v = torch.tensor([e[1] for e in train_pos_idx], dtype=torch.long)
    scores = (z[u] * z[v]).sum(dim=1)

    # Loss: push normal edge scores toward 1 (maximize similarity)
    loss = (1 - torch.sigmoid(scores)).mean()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        val_score = evaluate(model, data, val_pos_idx)
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} "
            f"| Val normal score: {val_score:.4f}")

## 11. Final test evaluation

In [ ]:
test_auc, test_ap = evaluate_test(model, data, test_normal_idx, test_attack_idx)
print(f"Test AUROC: {test_auc:.4f}")
print(f"Test AUPRC: {test_ap:.4f}")